# 04 — Threshold sweep & error analysis

Reads outputs from **`python -m src.pipeline.training_pipeline`** (from repo root). Interprets F1-optimal threshold, precision–recall trade-off, and error mix by segment.

In [ ]:
from pathlib import Path

import json
import pandas as pd
import matplotlib.pyplot as plt

M = Path("../reports/metrics")
need = [M / "threshold_table.csv", M / "threshold_summary.json", M / "error_analysis_detailed.csv"]
missing = [p for p in need if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Run training first from project root:\n"
        "  python -m src.pipeline.training_pipeline\n"
        f"Missing: {missing}"
    )
print("Metrics directory OK:", M.resolve())
%matplotlib inline

## Threshold sweep (test set)

Each point: classify positive if `P(churn) >= threshold`. Lower threshold → higher recall, lower precision.

In [ ]:
tbl = pd.read_csv(M / "threshold_table.csv")
display(tbl.round(4).head(8))
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(tbl["threshold"], tbl["precision"], "o-", label="precision", ms=4)
ax.plot(tbl["threshold"], tbl["recall"], "s-", label="recall", ms=4)
ax.plot(tbl["threshold"], tbl["f1"], "^-", label="f1", ms=4)
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_title("Precision / recall / F1 vs threshold")
plt.tight_layout()
plt.show()

In [ ]:
with open(M / "threshold_summary.json") as f:
    summ = json.load(f)
best_t = float(summ["best_threshold"])
idx = (tbl["threshold"] - best_t).abs().idxmin()
row = tbl.loc[[idx]]
print("F1-optimal threshold:", best_t)
print("At that threshold (approx):")
print(row.to_string(index=False))

## Confusion-style counts (from error summary)

In [ ]:
with open(M / "error_summary.json") as f:
    es = json.load(f)
counts = {k: int(es.get(k, 0)) for k in ["TN", "TP", "FP", "FN"]}
print(counts)
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(list(counts.keys()), list(counts.values()), color=["#2ecc71", "#3498db", "#f39c12", "#e74c3c"], edgecolor="black")
ax.set_ylabel("Count")
ax.set_title("Test-set outcomes at trained pipeline default predict (0.5) — see pipeline")
plt.tight_layout()
plt.show()

*Note: `error_summary` reflects the pipeline’s hard predictions on the test set (sklearn default 0.5 for binary). API inference uses the F1-tuned threshold from `threshold_summary.json`.*

## Errors by tenure group and contract

Where does the model confuse customers? Useful for targeted review (e.g. month-to-month + low tenure).

In [ ]:
seg_t = pd.read_csv(M / "error_by_tenure_group.csv", index_col=0)
seg_c = pd.read_csv(M / "error_by_contract.csv", index_col=0)
print("By tenure_group:")
display(seg_t)
print("By contract:")
display(seg_c)

In [ ]:
err = pd.read_csv(M / "error_analysis_detailed.csv")
print(err["error_type"].value_counts())
print("\nSample false positives (predict churn, actually stayed):")
display(err[err["error_type"] == "FP"].head(3)[["churn_probability", "true_label", "pred_label"]])
print("\nSample false negatives (predict stay, actually churned):")
display(err[err["error_type"] == "FN"].head(3)[["churn_probability", "true_label", "pred_label"]])

## Takeaways

1. **F1-optimal threshold** balances precision and recall when campaign costs are unknown; replace with a **budget cap** (e.g. top *k* scores) in a real workflow.
2. **FP** = unnecessary retention spend; **FN** = missed churners — tune threshold to business cost ratio.
3. Segment tables show **where** errors concentrate (often month-to-month); aligns with EDA.
4. Figures for README: run `python scripts/generate_figures.py` from repo root.